# Customer Analysis - RFM and Cohort

Online Retail dataset. In this notebook i am only looking at customers who have a CustomerID (the rest cant be tracked)

Plan:
1. load + filter data
2. RFM segments
3. cohort retention

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

## 1. Load data

In [ ]:
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['LineTotal'] = df['Quantity'] * df['UnitPrice']
df.head()

In [ ]:
# cancelled invoices start with C
cancellations = df['InvoiceNo'].str.startswith('C', na=False)

df_cust = df[df['CustomerID'].notnull()].copy()
df_cust_valid = df_cust[(df_cust['Quantity'] > 0) & (df_cust['UnitPrice'] > 0) & (~cancellations)].copy()

print("Total rows with CustomerID:", len(df_cust))
print("Valid rows with CustomerID:", len(df_cust_valid))
print("Unique customers:", df_cust_valid['CustomerID'].nunique())

## 2. RFM Analysis

- Recency = days since last purchase
- Frequency = number of invoices
- Monetary = total money spent

In [ ]:
# reference date = 1 day after last invoice
ref_date = df_cust_valid['InvoiceDate'].max() + dt.timedelta(days=1)
print("Reference date:", ref_date)

rfm = df_cust_valid.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (ref_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'LineTotal': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm = rfm[rfm['Monetary'] > 0]
rfm.head()

In [ ]:
rfm.describe()

Monetary has a very big max compared to the mean, some customers spend a lot (outliers?)

### RFM scores (1-5)
For recency lower is better so labels are reversed

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# frequency has lots of same values so qcut gives error, using rank first
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm.head(10)

In [ ]:
def segment_customer(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    rf = (r + f) / 2
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent / New'
    elif r >= 3 and f <= 2:
        return 'Promising / Potential'
    elif r == 2 and f >= 3:
        return 'Needs Attention'
    elif r == 2 and f <= 2:
        return 'About To Sleep'
    elif r == 1 and f >= 4:
        return 'Cant Lose Them'
    elif r == 1 and f >= 2:
        return 'At Risk'
    else:
        return 'Hibernating / Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['Segment'].value_counts()

In [ ]:
segment_summary = rfm.groupby('Segment').agg(
    CustomerCount=('CustomerID', 'count'),
    AvgRecency=('Recency', 'mean'),
    AvgFrequency=('Frequency', 'mean'),
    AvgMonetary=('Monetary', 'mean'),
    TotalMonetary=('Monetary', 'sum')
).reset_index()

total_m = rfm['Monetary'].sum()
segment_summary['SpendShare%'] = segment_summary['TotalMonetary'] / total_m * 100
segment_summary['CustomerShare%'] = segment_summary['CustomerCount'] / len(rfm) * 100
segment_summary = segment_summary.sort_values(by='TotalMonetary', ascending=False)

segment_summary

Champions are only a small part of customers but give the biggest share of spend. 
Hibernating / Lost is a big group but low spend.

## 3. Cohort Analysis

Cohort = month of the customer's first purchase. Then check how many of them come back in later months.

In [ ]:
df_cust_valid['InvoiceMonth'] = df_cust_valid['InvoiceDate'].dt.to_period('M')
df_cust_valid['CohortMonth'] = df_cust_valid.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

def get_cohort_index(df_sub):
    year_diff = df_sub['InvoiceMonth'].dt.year - df_sub['CohortMonth'].dt.year
    month_diff = df_sub['InvoiceMonth'].dt.month - df_sub['CohortMonth'].dt.month
    return year_diff * 12 + month_diff + 1

df_cust_valid['CohortIndex'] = get_cohort_index(df_cust_valid)
df_cust_valid.head()

In [ ]:
cohort_data = df_cust_valid.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_counts = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_counts

In [ ]:
cohort_sizes = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_sizes, axis=0) * 100
print(retention.round(1))

## Notes

- retention falls a lot after the first month
- Dec 2010 cohort has best retention
- NaN in the table is because later cohorts dont have enough months of data

Todo: make heatmap for retention, right now its just a table